# What happened this week at ARM's Bankhead National Forest site?

**A seven-day survey of the BNF observatory, 24–30 July 2026**

*Scott Collis and Claude (Anthropic)*

Most ARM analysis starts from a known event. This notebook starts from the opposite
position — *something* happened at a site over the past week, and we do not yet know
what, which instruments were running, or which hour deserves a closer look.

The workflow is deliberately ordered:

1. **Discover** which datastreams actually have data — the ARM catalog API is unusable, so we probe.
2. **Stage** the lightweight surface and profiling streams in bulk (~2.5 GB for a week).
3. **Read** the week off them and let the surface data nominate the interesting hours.
4. **Only then** stage scanning radar, targeted at that window.

That ordering is the point. A week of BNF C-SAPR2 is 1511 files at ~700 MB per volume;
inverted, this notebook would download a terabyte to look at one afternoon of weather.

The week splits cleanly: a wet, convectively active start under a moist airmass, then a
hot, dry ridge from 26 July with no measurable rain at any facility after 27 July 11:34 UTC.
The convective line of 24 July gets the radar treatment in section 7.

| Source | Product | Access |
|---|---|---|
| Surface meteorology, 4 facilities | `bnfmet{M1,S20,S30,S40}.b1` | ARM Live web service |
| Parsivel2 disdrometers | `bnfld*.b1`, `bnfldquants*.c1` | ARM Live web service |
| Ceilometer / SIRS / MWR | `bnfceilM1.b1`, `bnfsirsM1.b1`, `bnfmwr3cM1.b1` | ARM Live web service |
| Radiosondes + PBL VAP | `bnfsondewnpnM1.b1`, `bnfpblhtsonde1mcfarlM1.c1` | ARM Live web service |
| C-SAPR2 dual-pol radar | `bnfcsapr2cfrS3.a1` — interleaved PPI + RHI | ARM Live web service |

## 1. Environment and configuration

`cmweather` supplies the radar colormaps (`ChaseSpectral` for reflectivity, `balance`
for signed fields, `CM_rhohv` for correlation coefficient).

Two matplotlib settings below are load-bearing; each is explained where it is set.

In [ ]:
import os, json, glob, time, pathlib, urllib.parse, urllib.request
import concurrent.futures as cf
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import TwoSlopeNorm
import cartopy, cartopy.crs as ccrs
import cartopy.feature as cfeature
import pyart, cmweather   # noqa: F401 -- cmweather registers colormaps on import

# --- GOTCHA 1 -------------------------------------------------------------
# savefig.bbox='tight' silently mis-crops cartopy GeoAxes: panels placed at
# explicit figure rectangles get dropped from the saved PNG. Force standard.
mpl.rcParams["savefig.bbox"] = "standard"

# --- GOTCHA 2 -------------------------------------------------------------
# Cartopy caches shapefiles inside its package directory; where that is
# read-only (containers, shared installs) the first coastline draw raises
# PermissionError. Redirect to a writable path.
cartopy.config["data_dir"] = os.environ.get(
    "CARTOPY_DATA_DIR", str(pathlib.Path.home() / ".local/share/cartopy"))

mpl.rcParams.update({"figure.dpi": 110, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
# ---- Configuration -------------------------------------------------------
SITE       = "bnf"
FACILITIES = ["M1", "S3", "S4", "S20", "S30", "S40"]
START, END = "2026-07-24", "2026-07-31"     # ARM Live treats END as exclusive
DATA_DIR   = pathlib.Path(os.environ.get("BNF_DATA_DIR", "~/data/bnf_week")).expanduser()
FIG_DIR    = pathlib.Path("BNF_week_survey_outputs")
for d in (DATA_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

ARM_QUERY_URL = "https://adc.arm.gov/armlive/data/query"
ARM_SAVE_URL  = "https://adc.arm.gov/armlive/data/saveData"

# Facility colours, threaded consistently through every figure below.
FAC_COLOR = {"M1": "#1b3a6b", "S20": "#c1666b", "S30": "#3f8f6b", "S40": "#d99a2b"}

## 2. Credentials

An ARM Live account is required (free, at [adc.arm.gov](https://adc.arm.gov/armlive/)).
Credentials are read from the environment and nothing is written into this notebook.

```bash
export ARM_USER=yourname
export ARM_TOKEN=your_token
```

**Note the shape of the API.** ARM Live passes credentials as a query parameter —
`?user=<name>:<token>` — so the token ends up inside every request URL. A traceback
from `urllib` prints that URL. Keep outputs cleared when committing a notebook that
touches this service, and never build the URL in a cell you intend to leave executed.
The helpers below construct URLs internally and never return or print them.

In [ ]:
def arm_credentials():
    """Return (user, token) for the ARM Live web service from the environment.

    Never hardcode these, and never echo the return value: the token is a
    bearer credential for the whole ARM archive.
    """
    user  = os.environ.get("ARM_USER")  or os.environ.get("ARMUNAME")
    token = os.environ.get("ARM_TOKEN") or os.environ.get("ALIVE_TOKEN")
    if not user or not token:
        raise RuntimeError(
            "Set ARM_USER and ARM_TOKEN in the environment "
            "(register at https://adc.arm.gov/armlive/)")
    return user, token


def arm_list_files(datastream, start=None, end=None):
    """Filenames in a datastream between two YYYY-MM-DD dates. [] if none exist.

    A datastream that does not exist returns an empty list rather than an
    error, which is what makes the discovery sweep in section 3 practical.
    """
    user, token = arm_credentials()
    query = urllib.parse.urlencode({"user": f"{user}:{token}", "ds": datastream,
                                    "start": start or START, "end": end or END,
                                    "wt": "json"})
    with urllib.request.urlopen(f"{ARM_QUERY_URL}?{query}", timeout=60) as resp:
        return sorted(json.load(resp).get("files", []))


class ArmFetchError(RuntimeError):
    """An ARM Live request failed, with the credential-bearing URL stripped out.

    urllib puts the failing URL into HTTPError.__str__, and ARM Live carries the
    token in that URL. Raising through this class keeps the token out of
    tracebacks, logs, and any notebook output that gets committed.
    """


def arm_fetch(fname, dest_dir, attempts=3):
    """Download one ARM file unless already present. Returns (Path, was_cached).

    Downloads to a .part file and renames on completion, so an interrupted run
    never leaves a truncated netCDF that reads as corrupt later. Transient TLS
    resets are common on long staging runs, hence the retries.
    """
    user, token = arm_credentials()
    target = pathlib.Path(dest_dir) / fname
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and target.stat().st_size > 0:
        return target, True
    query = urllib.parse.urlencode({"user": f"{user}:{token}", "file": fname})
    tmp = target.with_suffix(target.suffix + ".part")
    for attempt in range(1, attempts + 1):
        try:
            urllib.request.urlretrieve(f"{ARM_SAVE_URL}?{query}", tmp)
            tmp.rename(target)
            return target, False
        except Exception as exc:                      # noqa: BLE001
            tmp.unlink(missing_ok=True)
            if attempt == attempts:
                # Report the exception TYPE only -- str(exc) can embed the URL.
                raise ArmFetchError(
                    f"{fname}: {type(exc).__name__} after {attempts} attempts") from None
            time.sleep(2 ** attempt)

## 3. Discovery: the catalog is not queryable, so probe

ARM publishes a data-discovery interface, but neither entry point is usable
programmatically: `adc.arm.gov/discovery/rest/api/datastreams` returns the HTML
shell of a single-page app, and the Solr endpoint behind it answers `502`.

The method that works is to construct candidate datastream names and ask the Live
query endpoint about each one. Names follow `<site><instrument><facility>.<level>`,
so a few dozen instruments across six facilities and five data levels is on the order
of a thousand candidates — a minute of parallel requests.

**A ~3% hit rate is normal**, not a sign the sweep is malformed. Most combinations
describe instruments that were never deployed at that facility.

In [ ]:
COMMON_INSTRUMENTS = [
    "met", "aosmet", "ceil", "ld", "ldquants", "sirs", "sebs", "ecorsf", "stamp",
    "mwr3c", "mwrlos", "mplpolfs", "mfrsr7nch", "sondewnpn", "pblhtsonde1mcfarl",
    "interpolatedsonde", "dlfpt", "dlprofwind4news", "wbpluvio2", "vdis",
    "aoscpcf", "aosccn2cola", "aosnephdry", "aosaps", "aoso3", "qcrad1long",
    "tsiskycover", "irt", "smos", "rwpwindcon", "kazrcfrge", "csapr2cfr",
]
COMMON_LEVELS = ["b1", "c1", "a1", "s1", "b0"]


def discover_datastreams(site, facilities, instruments=None, levels=None, workers=16):
    """Probe constructed datastream names; return {name: n_files} for the hits."""
    instruments = instruments or COMMON_INSTRUMENTS
    levels = levels or COMMON_LEVELS
    names = [f"{site}{inst}{fac}.{lvl}"
             for inst in instruments for fac in facilities for lvl in levels]
    names = list(dict.fromkeys(names))

    def probe(ds):
        try:
            return ds, len(arm_list_files(ds))
        except Exception:
            return ds, -1          # transient failure, not "does not exist"

    with cf.ThreadPoolExecutor(workers) as pool:
        results = list(pool.map(probe, names))
    return {ds: n for ds, n in sorted(results) if n > 0}


available = discover_datastreams(SITE, FACILITIES)
print(f"{len(available)} datastreams with data, {START} to {END}\n")
for ds, n in available.items():
    print(f"  {ds:36s} {n:5d} files")

## 4. Staging: split the download by file size, not by curiosity

Surface and profiling streams run 1–30 MB per day. Scanning radar is ~700 MB per
volume. Staging them in one pass means waiting on gigabytes before knowing whether
anything happened, so they get separate passes with the radar deferred to section 7.

`arm_fetch` skips files already on disk, so an interrupted run resumes cheaply.
Expect ~200 files / 2.5 GB and roughly twelve minutes on eight threads for a week
of the full AMF surface loadout.

In [ ]:
# Stage only what sections 5-6 actually read. Discovery above lists everything
# the site published; staging all of it would pull high-rate lidar and aerosol
# streams (100+ files each) that this notebook never opens. Widen deliberately.
WANTED = [f"{SITE}met{f}.b1" for f in ["M1", "S20", "S30", "S40"]] + [
    f"{SITE}ldquantsM1.c1", f"{SITE}ldquantsS30.c1",
    f"{SITE}ceilM1.b1", f"{SITE}sirsM1.b1", f"{SITE}mwr3cM1.b1",
    f"{SITE}sondewnpnM1.b1", f"{SITE}pblhtsonde1mcfarlM1.c1",
]
LIGHT_STREAMS = [ds for ds in WANTED if ds in available]
missing = [ds for ds in WANTED if ds not in available]
if missing:
    print(f"not published this week, skipping: {missing}")


def stage_datastreams(datastreams, workers=8):
    """Download whole datastreams into DATA_DIR/<datastream>/."""
    jobs = [(ds, f) for ds in datastreams for f in arm_list_files(ds)]

    def fetch(job):
        ds, fname = job
        path, cached = arm_fetch(fname, DATA_DIR / ds)
        return ds, path, cached

    staged = {ds: [] for ds in datastreams}
    with cf.ThreadPoolExecutor(workers) as pool:
        for ds, path, _ in pool.map(fetch, jobs):
            staged[ds].append(path)
    return {ds: sorted(paths) for ds, paths in staged.items()}


staged = stage_datastreams(LIGHT_STREAMS)
total_mb = sum(p.stat().st_size for paths in staged.values() for p in paths) / 1e6
print(f"{sum(len(v) for v in staged.values())} files, {total_mb:.0f} MB")

## 5. Reading the week

`load_datastream` concatenates a stream's daily files and applies each variable's
companion `qc_<var>` flag, keeping only `qc == 0`. Two habits matter here:

- **QC flags are per-variable.** Filtering a whole frame on one variable's flag
  silently discards good data from the others, so each column is masked separately.
- **Reindex onto a regular grid before any cumulative sum.** A missing day otherwise
  draws as a straight interpolated line across the gap, which reads as steady
  accumulation rather than absent data.

In [ ]:
def load_datastream(datastream, variables, apply_qc=True, freq=None):
    """Concatenate a datastream's daily files into a QC-masked DataFrame."""
    files = sorted(glob.glob(str(DATA_DIR / datastream / "*")))
    if not files:
        raise FileNotFoundError(f"nothing staged for {datastream}")
    data = xr.open_mfdataset(files, combine="nested", concat_dim="time",
                             drop_variables=["base_time", "time_offset"])
    columns = {}
    for var in variables:
        if var not in data:
            continue
        arr = data[var]
        qc_name = f"qc_{var}"
        if apply_qc and qc_name in data:
            arr = arr.where(data[qc_name] == 0)     # per-variable, not per-frame
        columns[var] = arr.to_series()
    data.close()
    frame = pd.DataFrame(columns).sort_index()
    if freq is not None:
        # Gaps must stay gaps: reindexing makes a missing day visible rather
        # than letting cumsum() interpolate straight through it.
        frame = frame.reindex(pd.date_range(frame.index.min(), frame.index.max(), freq=freq))
    return frame


MET_VARS = ["temp_mean", "rh_mean", "wspd_arith_mean", "wdir_vec_mean",
            "tbrg_precip_total_corr"]
met  = {f: load_datastream(f"{SITE}met{f}.b1", MET_VARS, freq="1min")
        for f in ["M1", "S20", "S30", "S40"]}
ldq  = {f: load_datastream(f"{SITE}ldquants{f}.c1",
                           ["rain_rate", "mass_weighted_mean_diameter"], apply_qc=False)
        for f in ["M1", "S30"]}
ceil = load_datastream(f"{SITE}ceilM1.b1", ["first_cbh"])
sirs = load_datastream(f"{SITE}sirsM1.b1", ["down_short_hemisp", "down_long_hemisp1"])

# MWR: keep zenith-pointing samples only -- off-zenith scans bias the retrieval.
mwr_raw = load_datastream(f"{SITE}mwr3cM1.b1", ["lwp", "pwv", "elevation"], apply_qc=False)
mwr = mwr_raw[mwr_raw.elevation > 85][["lwp", "pwv"]].resample("5min").mean()

precip = pd.DataFrame({f: d.tbrg_precip_total_corr.fillna(0) for f, d in met.items()})
print(precip.sum().round(1).to_string())

### Facility geometry

Facility identity is not obvious from the datastream name. M1 is the main site;
S-numbered facilities are supplementary and can sit tens of kilometres away. Read
the coordinates out of the files before interpreting any spatial difference — at BNF
the four met stations span 58 km, which is exactly why the weekly rain totals differ
by a factor of four.

In [ ]:
def facility_geometry(facilities, ref_lon, ref_lat):
    """Range (km) and bearing (deg) of each met facility from a reference point."""
    rows = {}
    for fac in facilities:
        files = sorted(glob.glob(str(DATA_DIR / f"{SITE}met{fac}.b1" / "*")))
        with xr.open_dataset(files[0]) as data:
            lon = float(np.asarray(data["lon"].values).ravel()[0])
            lat = float(np.asarray(data["lat"].values).ravel()[0])
        dx = (lon - ref_lon) * 111.32 * np.cos(np.deg2rad(ref_lat))
        dy = (lat - ref_lat) * 110.57
        rows[fac] = {"lon": lon, "lat": lat, "range_km": np.hypot(dx, dy),
                     "bearing_deg": np.degrees(np.arctan2(dx, dy)) % 360}
    return pd.DataFrame(rows).T


RADAR_LON, RADAR_LAT = -87.13312, 34.63081      # C-SAPR2 at BNF S3
geometry = facility_geometry(["M1", "S20", "S30", "S40"], RADAR_LON, RADAR_LAT)
print(geometry.round(2).to_string())

In [ ]:
# Radiosonde-derived PBL depth: one profile per launch, so read the files
# individually rather than concatenating along a shared time axis.
pbl_rows = []
for path in sorted(glob.glob(str(DATA_DIR / f"{SITE}pblhtsonde1mcfarlM1.c1" / "*"))):
    with xr.open_dataset(path) as sonde:
        pbl_rows.append({
            "time": pd.Timestamp(sonde.time.values[0]),
            "pbl_richardson": float(sonde.pbl_height_bulk_richardson_pt5.values.ravel()[0]),
            "pbl_liu_liang":  float(sonde.pbl_height_liu_liang.values.ravel()[0]),
        })
pbl = pd.DataFrame(pbl_rows).set_index("time").sort_index()
print(f"{len(pbl)} sonde launches, PBL depth "
      f"{pbl.pbl_richardson.min():.0f}-{pbl.pbl_richardson.max():.0f} m")

## 6. The week in one figure

Six panels on a shared time axis. Placement is by explicit figure rectangle rather
than `tight_layout` or `constrained_layout` — those fight a shared-x stack and, with
cartopy axes anywhere in the figure, can displace panels at draw time.

Facility series are labelled with coloured text inside each panel instead of a legend
box, and day gridlines run on every panel so an event lines up vertically across all six.

In [ ]:
fig = plt.figure(figsize=(11.0, 10.6))
heights, axes, y = [0.148, 0.128, 0.118, 0.118, 0.128, 0.118], [], 0.945
for h in heights:
    y -= h
    axes.append(fig.add_axes([0.075, y, 0.845, h]))
    y -= 0.023
ax_t, ax_p, ax_w, ax_r, ax_c, ax_l = axes

for fac, frame in met.items():
    ax_t.plot(frame.index, frame.temp_mean, lw=0.8, color=FAC_COLOR[fac])
    ax_p.plot(frame.index, frame.tbrg_precip_total_corr.fillna(0).cumsum(),
              lw=1.4, color=FAC_COLOR[fac])
ax_t.set_ylabel("Air temp (°C)")
ax_p.set_ylabel("Cumulative\nrain (mm)")
ax_t.set_title("Bankhead National Forest, 24–30 July 2026: a wet, convectively "
               "active start giving way to a hot, dry ridge")
for i, fac in enumerate(FAC_COLOR):
    ax_t.text(1.004, 0.86 - 0.13 * i, fac, transform=ax_t.transAxes,
              color=FAC_COLOR[fac], fontsize=8, va="center")
for i, fac in enumerate(sorted(FAC_COLOR, key=lambda f: -precip[f].sum())):
    ax_p.text(0.012, 0.93 - 0.115 * i, f"{fac}  {precip[fac].sum():.0f} mm",
              transform=ax_p.transAxes, color=FAC_COLOR[fac], fontsize=7.5, va="top")

ax_p2 = ax_p.twinx()
ax_p2.plot(ldq["S30"].index, ldq["S30"].rain_rate, lw=0.5, color="#8a8a8a", alpha=0.85, zorder=0)
ax_p2.set_ylabel("S30 rain rate\n(mm h$^{-1}$)", color="#8a8a8a")
ax_p2.tick_params(colors="#8a8a8a"); ax_p2.set_ylim(0, 60)
ax_p2.spines["right"].set_color("#8a8a8a")

for fac in ["M1", "S40"]:
    ax_w.plot(met[fac].index, met[fac].wspd_arith_mean, lw=0.7, color=FAC_COLOR[fac])
ax_w.set_ylabel("Wind speed\n(m s$^{-1}$)")
ax_w.text(0.012, 0.90, "M1", transform=ax_w.transAxes, fontsize=7.5, color=FAC_COLOR["M1"], va="top")
ax_w.text(0.042, 0.90, "S40", transform=ax_w.transAxes, fontsize=7.5, color=FAC_COLOR["S40"], va="top")

ax_r.plot(sirs.index, sirs.down_short_hemisp, lw=0.6, color="#d99a2b")
ax_r.set_ylabel("Downwelling\nSW (W m$^{-2}$)")

ax_c.plot(ceil.index, ceil.first_cbh / 1000., ls="none", marker=".", ms=1.0,
          color="#4a4a4a", alpha=0.45)
ax_c.plot(pbl.index, pbl.pbl_richardson / 1000., marker="o", ms=4, lw=1.0, color="#c1666b")
ax_c.set_ylabel("Height (km AGL)"); ax_c.set_ylim(0, 8)
ax_c.text(0.012, 0.93, "ceilometer cloud base", transform=ax_c.transAxes,
          fontsize=7.5, color="#4a4a4a", va="top")
ax_c.text(0.012, 0.80, "radiosonde PBL top", transform=ax_c.transAxes,
          fontsize=7.5, color="#c1666b", va="top")

ax_l.plot(mwr.index, mwr.pwv, lw=0.9, color="#1b3a6b")
ax_l.set_ylabel("PWV (cm)")
ax_l2 = ax_l.twinx()
ax_l2.plot(mwr.index, mwr.lwp.clip(lower=0), lw=0.6, color="#3f8f6b", alpha=0.85)
ax_l2.set_ylabel("LWP (mm)", color="#3f8f6b")
ax_l2.tick_params(colors="#3f8f6b"); ax_l2.spines["right"].set_color("#3f8f6b")

for ax in axes:
    ax.set_xlim(pd.Timestamp(START), pd.Timestamp(END))
    ax.xaxis.set_major_locator(mdates.DayLocator())
    ax.xaxis.set_minor_locator(mdates.HourLocator(byhour=[6, 12, 18]))
    ax.grid(axis="x", color="0.9", lw=0.6)
    if ax is not axes[-1]:
        ax.set_xticklabels([])
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
axes[-1].set_xlabel("UTC")
fig.savefig(FIG_DIR / "bnf_week_overview.png", dpi=200)

### Daily summary

Cloudy fraction from the ceilometer and integrated shortwave from SIRS are the two
columns that separate the week into regimes at a glance: 0.91 cloudy and 7.4 MJ m⁻²
on the 24th against 0.04 and 24.4 MJ m⁻² on the 28th.

In [ ]:
temps = pd.concat([d.temp_mean for d in met.values()], axis=1)
daily = pd.DataFrame({
    "T_max_C": temps.max(axis=1).resample("1D").max(),
    "T_min_C": temps.min(axis=1).resample("1D").min(),
    **{f"rain_{fac}_mm": precip[fac].resample("1D").sum() for fac in precip},
    "peak_rainrate_S30_mmh": ldq["S30"].rain_rate.resample("1D").max(),
    "SW_peak_Wm2": sirs.down_short_hemisp.resample("1D").max(),
    "SW_daily_MJm2": sirs.down_short_hemisp.clip(lower=0).resample("1D").sum() * 60 / 1e6,
    "PWV_mean_cm": mwr.pwv.resample("1D").mean(),
    "LWP_max_mm": mwr.lwp.resample("1D").max(),
    "cloudy_frac": ceil.first_cbh.notna().resample("1D").mean(),
})
daily.index = daily.index.date
daily.to_csv(FIG_DIR / "bnf_daily_summary.csv", float_format="%.2f")
daily.round(2)

In [ ]:
# Which hours actually had rain? This is what nominates the radar window.
hourly = precip.resample("1h").sum()
wet_hours = hourly[hourly.sum(axis=1) > 0.2]
last_tip = precip[precip.sum(axis=1) > 0].index.max()
print(f"{len(wet_hours)} wet hours; last measurable tip at any facility: {last_tip} UTC\n")
print(wet_hours.head(12).round(2).to_string())

## 7. The 24 July convective line

The surface data nominates 15:30–19:00 UTC on 24 July: S40 took 43 of its 51 weekly
millimetres in that window and the S30 disdrometer peaked at 47 mm h⁻¹.

BNF publishes **one interleaved datastream** — `bnfcsapr2cfrS3.a1` mixes 15-sweep PPI
volumes with single-sweep RHIs. There are no separate `cfrppi` / `cfrhsrhi` streams as
at other ARM sites, so guessing those names returns nothing. The two scan types are
separated on file size (~700 MB versus ~20 MB) without opening each one.

Even inside the window we subsample: alternating volume pairs at ~16 min cadence
resolve the evolution as well as every scan does, for half the download.

In [ ]:
RADAR_DS   = f"{SITE}csapr2cfrS3.a1"
RADAR_DIR  = DATA_DIR / "csapr2"
WINDOW     = (pd.Timestamp("2026-07-24 15:30"), pd.Timestamp("2026-07-24 19:00"))
PPI_RHI_SPLIT_BYTES = 300_000_000       # PPI volumes ~700 MB, RHIs ~20 MB


def radar_file_time(fname):
    """Parse the YYYYMMDD.HHMMSS stamp out of an ARM radar filename."""
    parts = fname.split(".")
    return pd.Timestamp(f"{parts[-3]}{parts[-2]}")


all_radar = arm_list_files(RADAR_DS)
in_window = [f for f in all_radar if WINDOW[0] <= radar_file_time(f) <= WINDOW[1]]
# Files alternate RHI, PPI, RHI, PPI...; keep every other pair.
pairs    = [in_window[i:i + 2] for i in range(0, len(in_window), 2)]
selected = [f for pair in pairs[::2] for f in pair]
print(f"{len(all_radar)} radar files this week; {len(in_window)} in the window; "
      f"staging {len(selected)}")

with cf.ThreadPoolExecutor(6) as pool:
    list(pool.map(lambda f: arm_fetch(f, RADAR_DIR), selected))

staged_radar = sorted(RADAR_DIR.glob("*.nc"))
ppi_files = [p for p in staged_radar if p.stat().st_size >= PPI_RHI_SPLIT_BYTES]
rhi_files = [p for p in staged_radar if p.stat().st_size <  PPI_RHI_SPLIT_BYTES]
print(f"{len(ppi_files)} PPI volumes, {len(rhi_files)} RHIs, "
      f"{sum(p.stat().st_size for p in staged_radar) / 1e9:.1f} GB")

### Storm evolution: growth versus intensification

Echo area is computed on the native gate geometry. Each gate cell covers
`(r · Δφ) · Δr`, so cells tile the sweep without overlap and no Cartesian regridding
is needed — which also means these numbers are independent of any map projection,
a point that matters in the next section.

In [ ]:
def sweep_echo_areas(path, thresholds=(20, 30, 40, 50)):
    """Range-weighted echo area (km^2) above each threshold on the lowest sweep."""
    radar = pyart.io.read(path, include_fields=["reflectivity"])
    sweep = radar.extract_sweeps([0])
    zh = sweep.fields["reflectivity"]["data"]
    gx, gy, _ = sweep.get_gate_x_y_z(0)
    range_km = np.hypot(gx, gy) / 1000.0
    dr  = float(np.diff(radar.range["data"])[0]) / 1000.0
    daz = np.deg2rad(float(np.median(np.abs(np.diff(sweep.azimuth["data"][:50])))))
    cell_km2 = (range_km * daz) * dr
    valid = ~np.ma.getmaskarray(zh)
    row = {"time": pyart.util.datetime_from_radar(radar),
           "max_dbz": float(np.ma.max(radar.fields["reflectivity"]["data"]))}
    for thresh in thresholds:
        row[f"area_ge{thresh}_km2"] = float(cell_km2[valid & (zh >= thresh)].sum())
    del radar, sweep
    return row


evolution = pd.DataFrame([sweep_echo_areas(p) for p in ppi_files]).set_index("time").sort_index()
growth = evolution["area_ge20_km2"].iloc[-1] / evolution["area_ge20_km2"].iloc[0]
print(f"20 dBZ area grew {growth:.0f}x; peak Z_H stayed within "
      f"{evolution.max_dbz.min():.1f}-{evolution.max_dbz.max():.1f} dBZ")
evolution.round(1)

### Hail or heavy rain?

A 60 dBZ core at C band invites a hail reading, so test it rather than assuming.
Hail tumbles and depolarises — low Z_DR, depressed ρ_HV. Large oblate raindrops give
high Z_DR with ρ_HV near unity.

In [ ]:
def dualpol_gatefilter(radar, snr_min=10.0, rhohv_min=0.85, z_min=5.0, speckle=40):
    """Conservative gate filter for C-band dual-pol precipitation analysis.

    despeckle_field is a module-level function that RETURNS a GateFilter --
    it is not a GateFilter method.
    """
    gatefilter = pyart.correct.despeckle_field(radar, "reflectivity", size=speckle)
    gatefilter.exclude_masked("reflectivity")
    gatefilter.exclude_below("signal_to_noise_ratio_copolar_h", snr_min)
    gatefilter.exclude_below("copol_correlation_coeff", rhohv_min)
    gatefilter.exclude_below("reflectivity", z_min)
    gatefilter.exclude_below("normalized_coherent_power", 0.4)
    return gatefilter


peak_ppi = ppi_files[int(np.argmax(evolution["area_ge40_km2"].values))]
radar = pyart.io.read(peak_ppi)
gatefilter = dualpol_gatefilter(radar)

zh   = radar.get_field(0, "reflectivity")
zdr  = radar.get_field(0, "differential_reflectivity")
rho  = radar.get_field(0, "copol_correlation_coeff")
core = (zh >= 50) & ~np.ma.getmaskarray(zh)
hail_like = core & (zdr < 1.0) & (rho < 0.95)
print(f"{core.sum()} gates >= 50 dBZ\n"
      f"  median Z_DR   {np.ma.median(zdr[core]):.2f} dB\n"
      f"  median rho_HV {np.ma.median(rho[core]):.3f}\n"
      f"  hail-like     {100 * hail_like.sum() / core.sum():.0f}%")

### GOTCHA 3: do not let Py-ART choose the map projection

`pyart.graph.RadarMapDisplay.plot_ppi_map` builds a cartopy `AzimuthalEquidistant`
axes centred on the radar. **That transform is inaccurate at its own origin when
`central_latitude != 0`** — it maps the radar's true position to roughly `(0, +21 km)`
in projection space instead of `(0, 0)`, so every gate is drawn about 21 km south of
where it belongs.

The tell is that the radar marker sits north of its own clutter fan. It is not a data
problem: Py-ART's `gate_longitude` / `gate_latitude` are correct spherical-earth
coordinates. Only the axes projection is wrong.

The cell below measures the offset for this radar rather than assuming 21 km — the
error scales with latitude.

In [ ]:
def aeqd_origin_offset(radar):
    """Metres by which a radar-centred cartopy AEQD misplaces the radar itself."""
    lon = float(radar.longitude["data"][0])
    lat = float(radar.latitude["data"][0])
    aeqd = ccrs.AzimuthalEquidistant(central_longitude=lon, central_latitude=lat)
    return aeqd.transform_point(lon, lat, ccrs.PlateCarree())


dx, dy = aeqd_origin_offset(radar)
print(f"Radar-centred AEQD places the radar at ({dx:.0f}, {dy:.0f}) m "
      f"instead of (0, 0)\n  -> gates would be drawn {abs(dy)/1000:.1f} km off")

# Py-ART's own gate coordinates, by contrast, are correct.
sweep_slice = radar.get_slice(0)
gate_lon = radar.gate_longitude["data"][sweep_slice]
gate_lat = radar.gate_latitude["data"][sweep_slice]
print(f"  gate[0,0] = ({gate_lon[0,0]:.5f}, {gate_lat[0,0]:.5f})")
print(f"  site      = ({RADAR_LON:.5f}, {RADAR_LAT:.5f})")

The fix is to drive the axes yourself: a **Mercator** projection, with gates and every
marker drawn through `PlateCarree` so no AEQD round-trip happens anywhere.

Verify it afterwards rather than trusting it — the near-range clutter centroid should
coincide with the site, and ground-site markers should land on the echoes their gauges
recorded.

In [ ]:
def ppi_gate_latlon(radar, field, sweep=0, gatefilter=None):
    """Gate lon/lat and masked field for one sweep, ready for pcolormesh."""
    sl = radar.get_slice(sweep)
    lon = radar.gate_longitude["data"][sl]
    lat = radar.gate_latitude["data"][sl]
    data = np.ma.masked_array(radar.fields[field]["data"][sl])
    if gatefilter is not None:
        data = np.ma.masked_where(gatefilter.gate_excluded[sl], data)
    return lon, lat, data


# Verification: near-range gates are dominated by the ground-clutter fan, whose
# centroid must coincide with the site if field and markers share a reference.
near = np.broadcast_to(radar.range["data"] / 1000.0 < 12.0, gate_lon.shape)
print(f"clutter centroid ({gate_lon[near].mean():.5f}, {gate_lat[near].mean():.5f})")
print(f"site             ({RADAR_LON:.5f}, {RADAR_LAT:.5f})")

In [ ]:
PANELS = [
    ("reflectivity",              "Reflectivity Z$_H$ (dBZ)",              -10, 65, "ChaseSpectral", False),
    ("differential_reflectivity", "Differential reflectivity Z$_{DR}$ (dB)", -2,  6, "balance",       True),
    ("copol_correlation_coeff",   "Correlation coefficient $\\rho_{HV}$",   0.8, 1.0, "CM_rhohv",   False),
    ("mean_doppler_velocity",     "Doppler velocity (m s$^{-1}$)",          -17, 17, "balance",       True),
]

fig = plt.figure(figsize=(11.4, 9.4))
rect_w, rect_h = 0.415, 0.335
aspect = (rect_w * 11.4) / (rect_h * 9.4)
half_lon = 1.22
half_lat = half_lon / aspect
extent = [-87.15 - half_lon, -87.15 + half_lon, 34.32 - half_lat, 34.32 + half_lat]
rects = [[0.055, 0.575, rect_w, rect_h], [0.545, 0.575, rect_w, rect_h],
         [0.055, 0.120, rect_w, rect_h], [0.545, 0.120, rect_w, rect_h]]
azimuth_rad = np.deg2rad(210.0)
ray_km = np.arange(0, 114, 1.0)

for (field, label, vmin, vmax, cmap, diverging), rect, letter in zip(PANELS, rects, "abcd"):
    # Mercator axes -- NOT a radar-centred AzimuthalEquidistant (GOTCHA 3).
    ax = fig.add_axes(rect, projection=ccrs.Mercator(central_longitude=RADAR_LON))
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    lon, lat, data = ppi_gate_latlon(radar, field, 0, gatefilter)
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax) if diverging else None
    mesh = ax.pcolormesh(lon, lat, data, transform=ccrs.PlateCarree(), cmap=cmap,
                         vmin=None if diverging else vmin,
                         vmax=None if diverging else vmax,
                         norm=norm, shading="auto", rasterized=True)
    ax.add_feature(cfeature.STATES.with_scale("10m"), lw=0.5, edgecolor="0.3")
    grid = ax.gridlines(draw_labels=True, lw=0.4, color="0.85",
                        x_inline=False, y_inline=False)   # or labels land in the map
    grid.top_labels = grid.right_labels = False
    grid.xlabel_style = grid.ylabel_style = {"size": 7}

    ax.plot(RADAR_LON + ray_km * np.sin(azimuth_rad) / (111.32 * np.cos(np.deg2rad(RADAR_LAT))),
            RADAR_LAT + ray_km * np.cos(azimuth_rad) / 110.57,
            transform=ccrs.PlateCarree(), color="k", lw=0.7, ls="--", alpha=0.55, zorder=5)
    for fac, row in geometry.iterrows():
        ax.plot(row.lon, row.lat, marker="o", ms=4, mfc="none", mec="k", mew=1.0,
                transform=ccrs.PlateCarree(), zorder=6)
        ax.text(row.lon + 0.035, row.lat + 0.03, fac, fontsize=7,
                transform=ccrs.PlateCarree(), zorder=6)
    ax.plot(RADAR_LON, RADAR_LAT, marker="^", ms=7, color="k",
            transform=ccrs.PlateCarree(), zorder=7)
    ax.text(RADAR_LON + 0.045, RADAR_LAT + 0.045, "C-SAPR2 (S3)", fontsize=7.5,
            transform=ccrs.PlateCarree(), zorder=7)

    cax = fig.add_axes([rect[0] + 0.035, rect[1] - 0.055, rect[2] - 0.07, 0.014])
    bar = fig.colorbar(mesh, cax=cax, orientation="horizontal", extend="both")
    bar.set_label(label, fontsize=8); bar.ax.tick_params(labelsize=7)
    fig.text(rect[0] - 0.040, rect[1] + rect[3] + 0.005, letter,
             fontsize=11, fontweight="bold", va="bottom")

scan_time = pyart.util.datetime_from_radar(radar).strftime("%d %B %Y %H:%M:%S UTC")
fig.text(0.055, 0.968, f"BNF C-SAPR2, {scan_time}, 1.5° PPI: a mature convective "
         "line of heavy rain, not hail", fontsize=11, ha="left")
fig.text(0.055, 0.944, "Dashed line: the 210° RHI plane, which runs down the M1/S40 "
         "baseline. Open circles: surface facilities.", fontsize=8, ha="left", color="0.3")
fig.savefig(FIG_DIR / "bnf_csapr2_ppi_dualpol.png", dpi=190)

### Vertical structure: how deep did the updraft get?

The 40 dBZ echo top means little in isolation. Against the 0 °C and −20 °C levels from
the nearest pre-storm sounding it measures how far precipitation-sized particles were
lofted into the mixed-phase layer — the quantity that matters for both hail growth and
charge separation.

In [ ]:
def read_rhi(path, filtered=True):
    """Read a single-sweep RHI into plain arrays for pcolormesh."""
    scan = pyart.io.read(path)
    gate_filter = dualpol_gatefilter(scan) if filtered else None
    x, y, z = scan.get_gate_x_y_z(0)
    out = {"rng": np.hypot(x, y) / 1000.0, "ht": z / 1000.0,
           "time": pyart.util.datetime_from_radar(scan),
           "nyquist": float(scan.get_nyquist_vel(0))}
    for key, field in [("Z", "reflectivity"), ("D", "differential_reflectivity"),
                       ("V", "mean_doppler_velocity")]:
        data = np.ma.masked_array(scan.get_field(0, field))
        # Filter for measurement, not for display: an unfiltered Z_H panel keeps
        # the weak-echo anvil that the dual-pol filter discards.
        if gate_filter is not None and not (key == "Z" and not filtered):
            data = np.ma.masked_where(gate_filter.gate_excluded[scan.get_slice(0)], data)
        out[key] = data
    del scan
    return out


def echo_top(scan, threshold=40.0):
    """Highest gate (km) at or above a reflectivity threshold; NaN if absent."""
    mask = scan["Z"] >= threshold
    if np.ma.isMaskedArray(mask):
        mask = mask.filled(False)
    return float(scan["ht"][mask].max()) if mask.any() else float("nan")


def sounding_level(path, target_c, ref_alt_m):
    """Altitude (km above the reference) where the sounding first hits target_c."""
    with xr.open_dataset(path) as sonde:
        alt, tdry = sonde.alt.values, sonde.tdry.values
    return float(alt[np.argmin(np.abs(tdry - target_c))] - ref_alt_m) / 1000.0


sonde_files = sorted(glob.glob(str(DATA_DIR / f"{SITE}sondewnpnM1.b1" / "*20260724.16*")))
radar_alt   = float(radar.altitude["data"][0])
level_0c    = sounding_level(sonde_files[0],   0.0, radar_alt)
level_m20c  = sounding_level(sonde_files[0], -20.0, radar_alt)

tops = pd.DataFrame([{"time": (s := read_rhi(p))["time"], "echo_top_40dbz_km": echo_top(s)}
                     for p in rhi_files]).set_index("time").sort_index()
print(f"0 °C at {level_0c:.1f} km, -20 °C at {level_m20c:.1f} km (above radar)")
print(f"40 dBZ echo tops {tops.echo_top_40dbz_km.min():.1f}-"
      f"{tops.echo_top_40dbz_km.max():.1f} km  "
      f"({tops.echo_top_40dbz_km.isna().sum()} scans with no 40 dBZ gate in the plane)")

In [ ]:
peak_rhi = sorted(rhi_files)[int(np.nanargmax(tops.echo_top_40dbz_km.values))]
display_scan  = read_rhi(peak_rhi, filtered=False)   # raw Z_H for the picture
filtered_scan = read_rhi(peak_rhi, filtered=True)    # dual-pol filtered

fig = plt.figure(figsize=(11.0, 8.6))
axes = [fig.add_axes([0.075, 0.685, 0.79, 0.235]),
        fig.add_axes([0.075, 0.395, 0.79, 0.235]),
        fig.add_axes([0.075, 0.105, 0.79, 0.235])]
nyquist = display_scan["nyquist"]
rhi_panels = [("Z", "Reflectivity Z$_H$ (dBZ)", -10, 65, "ChaseSpectral", None),
              ("D", "Differential reflectivity Z$_{DR}$ (dB)", -2, 6, "balance",
               TwoSlopeNorm(vmin=-2, vcenter=0, vmax=6)),
              ("V", "Doppler velocity (m s$^{-1}$)", -nyquist, nyquist, "balance",
               TwoSlopeNorm(vmin=-nyquist, vcenter=0, vmax=nyquist))]

for ax, (key, label, vmin, vmax, cmap, norm) in zip(axes, rhi_panels):
    source = display_scan if key == "Z" else filtered_scan
    mesh = ax.pcolormesh(display_scan["rng"], display_scan["ht"], source[key],
                         cmap=cmap, vmin=None if norm else vmin,
                         vmax=None if norm else vmax, norm=norm,
                         shading="auto", rasterized=True)
    ax.set_ylim(0, 16); ax.set_xlim(0, 110)
    ax.set_ylabel("Height (km AGL)")
    ax.axhline(level_0c,   color="0.25", lw=0.8, ls="--")
    ax.axhline(level_m20c, color="0.25", lw=0.8, ls=":")
    cax = fig.add_axes([0.878, ax.get_position().y0, 0.014, 0.235])
    bar = fig.colorbar(mesh, cax=cax, extend="both")
    bar.set_label(label, fontsize=8); bar.ax.tick_params(labelsize=7)

axes[0].text(2, level_0c + 0.4,   "0 °C",   fontsize=7.5, color="0.25")
axes[0].text(2, level_m20c + 0.4, "−20 °C", fontsize=7.5, color="0.25")
axes[-1].set_xlabel("Range along 210° azimuth from C-SAPR2 (km)")
fig.text(0.075, 0.965, f"BNF C-SAPR2 RHI along 210°, "
         f"{display_scan['time'].strftime('%d %B %Y %H:%M:%S UTC')}", fontsize=11, ha="left")
fig.text(0.075, 0.941, "Panel a is unfiltered Z$_H$; b and c are dual-pol filtered. "
         f"Velocity is aliased at the ±{nyquist:.1f} m s$^{{-1}}$ Nyquist — abrupt "
         "colour reversals aloft are folding, not shear.", fontsize=7.8, ha="left", color="0.3")
fig.savefig(FIG_DIR / "bnf_csapr2_rhi.png", dpi=190)

## 8. What the week showed

**The regime split.** 24–25 July were wet under a moist airmass (PWV 5.8–5.9 cm). From
26 July a hot, dry ridge took over: PWV fell to 3.4 cm, ceilometer cloudy fraction
collapsed from 0.91 to 0.04, daily insolation rose from 7.4 to ~24 MJ m⁻², and peak
temperature reached 35.6 °C on the 28th. No facility recorded measurable rain after
27 July 11:34 UTC.

**Rainfall is spatially patchy.** Weekly totals ranged from 13 mm at S30 to 51 mm at
S40 — a factor of four across facilities separated by tens of kilometres, with 43 of
S40's 51 mm falling on 24 July alone. Any single-gauge summary of this week would be
wrong by a factor of several.

**The 24 July line grew rather than intensified.** Echo area above 20 dBZ grew 13-fold
between 15:32 and 18:45 while peak reflectivity oscillated in a 58.5–63.0 dBZ band with
no trend. Those are different claims about a storm, and the areal one is usually the
more informative.

**The cores were rain, not hail.** Median Z_DR of 3.3 dB with ρ_HV of 0.98 in the
Z_H ≥ 50 dBZ gates is the signature of large oblate raindrops; only 7% of core gates met
a hail criterion. The 40 dBZ echo top reached 7.4 km against a −20 °C level at 8.5 km,
so precipitation-sized particles entered the mixed-phase layer without reaching the
primary charging zone.

### Caveats

These are real limits of the measurement, not boilerplate:

- **No attenuation correction.** C-band Z_H and Z_DR are biased low behind a 60 dBZ
  core, visible as a negative Z_DR streak downrange. Correct before quantitative rain rates.
- **Velocity is aliased** at the ±16.3 m s⁻¹ Nyquist. Dealias before any kinematic work.
- **An RHI is a fixed azimuth.** Scan-to-scan echo-top changes partly reflect the line
  translating across the plane; PPI-derived areas are the better growth measure.
- **Echo tops are gappy** where no gate in the plane reached 40 dBZ. Those stay as gaps
  rather than being interpolated across.
- **Radar coverage has holes**: 25 July (22 files) and 27 July (47) are partial days and
  26 July has none. The 24 July analysis is unaffected.
- **Tipping buckets under-report light rain.** Where gauge and disdrometer disagree at
  low rates, trust the disdrometer for rate and the gauge for total.
- **MWR retrievals are unreliable during rain** — treat LWP spikes coincident with rain
  as artifacts rather than cloud liquid.

### Reuse

Point `SITE`, `FACILITIES` and the date range at another ARM site and sections 3–6 run
unchanged; the discovery sweep will find whatever that site actually publishes. Section 7
is BNF-specific only in the datastream name and the RHI azimuth.